In [9]:
import pandas as pd
import numpy as np
import xgboost as xgb
import time
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

file_path = '../data/raw/MZVAV-1.csv'

if not os.path.exists(file_path):
    print(f"ERROR: Cannot find {file_path}. Please check your folders!")
else:
    df = pd.read_csv(file_path)
    
    target = 'Fault Detection Ground Truth'
    
    df_clean = df.ffill() 
    
    X = df_clean.drop(target, axis=1)
    X = X.select_dtypes(include=[np.number]) # Only keeps the 16 numeric sensors [cite: 232]
    y = df_clean[target]

    split_idx = int(len(df_clean) * 0.8)
    X_train_raw, X_test_raw = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    model_xgb = xgb.XGBClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        max_depth=5, 
        random_state=42, 
        eval_metric='logloss'
    )
    
    print("All checks passed. Starting training on Apple M1...")
    start_t = time.time()
    model_xgb.fit(X_train, y_train)
    train_time = time.time() - start_t

    y_pred = model_xgb.predict(X_test)
    
    print(f"\n--- XGBOOST RESULTS FOR THESIS SECTION 4.2 ---")
    print(f"Training Time: {train_time:.4f} seconds")
    print(classification_report(y_test, y_pred))
    
    print(f"\nYour 16 Sensors are: {list(X.columns)}")

All checks passed. Starting training on Apple M1...

--- XGBOOST RESULTS FOR THESIS SECTION 4.2 ---
Training Time: 0.3069 seconds
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.80      0.89     54432

    accuracy                           0.80     54432
   macro avg       0.50      0.40      0.45     54432
weighted avg       1.00      0.80      0.89     54432


Your 16 Sensors are: ['AHU: Supply Air Temperature', 'AHU: Supply Air Temperature Set Point', 'AHU: Outdoor Air Temperature', 'AHU: Mixed Air Temperature', 'AHU: Return Air Temperature', 'AHU: Supply Air Fan Status', 'AHU: Return Air Fan Status', 'AHU: Supply Air Fan Speed Control Signal', 'AHU: Return Air Fan Speed Control Signal', 'AHU: Outdoor Air Damper Control Signal  ', 'AHU: Return Air Damper Control Signal', 'AHU: Cooling Coil Valve Control Signal', 'AHU: Heating Coil Valve Control Signal', 'AHU: Supply Air Duct Static Pressure 

/opt/homebrew/Cellar/jupyterlab/4.4.9/libexec/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Cellar/jupyterlab/4.4.9/libexec/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Cellar/jupyterlab/4.4.9/libexec/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"